# Invasive Species Detection — End-to-End Demo Pipeline

**This notebook runs entirely on synthetic, procedurally generated imagery.**
No real drone captures, confidential datasets, or Azure credentials are used
or required. It demonstrates the technical approach — preprocessing, tiling,
SAM-style segmentation, tree candidate extraction, crown separation,
duplicate merging, visual embeddings, and clustering — not production results.

See `../README.md` for the full confidentiality disclaimer.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np

from utils.synthetic_data import generate_synthetic_orthomosaic
from utils.visualization import show_image, overlay_crown_detections, plot_embedding_clusters
from preprocessing.image_preprocessing import preprocess_orthomosaic
from preprocessing.tiling import tile_image, tile_to_scene_coords
from segmentation.sam_segmentation import get_default_segmenter
from detection.tree_candidate_extraction import extract_tree_candidates
from detection.crown_separation import separate_merged_crowns
from detection.duplicate_merging import SceneDetection, merge_duplicate_detections
from clustering.visual_embeddings import get_default_embedder
from clustering.cluster_analysis import cluster_embeddings, project_to_2d
from utils.geo_utils import detections_to_geodataframe

## Step 1 — Generate a synthetic "drone orthomosaic"

Stands in for a real drone capture. Produces a fake RGB scene with randomly
scattered blob-shaped tree-crown regions on a bare-earth background.

In [ ]:
scene = generate_synthetic_orthomosaic(width=1536, height=1536, num_crowns=60, seed=7)
show_image(scene.image, title=f"Synthetic orthomosaic ({len(scene.crowns)} ground-truth crowns)")

## Step 2 — Preprocess

Denoise + CLAHE illumination correction.

In [ ]:
preprocessed = preprocess_orthomosaic(scene.image)
show_image(preprocessed, title="Preprocessed")

## Step 3 — Tile

Split into overlapping tiles so segmentation stays memory-safe on
large orthomosaics.

In [ ]:
tiles = tile_image(preprocessed, tile_size=512, overlap=64)
print(f"Generated {len(tiles)} tiles")
show_image(tiles[0].image, title="Example tile")

## Step 4 — Segment each tile

Uses the SAM-compatible segmenter interface. The public demo runs the
classical-CV fallback (`MockCrownSegmenter`) so no model weights are
required; swap in `SamCrownSegmenter(checkpoint_path=...)` for the
production path.

In [ ]:
segmenter = get_default_segmenter()

scene_detections = []
all_candidate_masks = []

for tile in tiles:
    masks = segmenter.segment(tile.image)
    candidates = extract_tree_candidates(masks)

    split_candidates = []
    for cand in candidates:
        split_candidates.extend(separate_merged_crowns(cand))

    for cand in split_candidates:
        scene_x, scene_y = tile_to_scene_coords(tile, cand.pixel_x, cand.pixel_y)
        scene_detections.append(
            SceneDetection(
                scene_x=scene_x,
                scene_y=scene_y,
                radius_px=cand.radius_px,
                shape_score=cand.shape_score,
            )
        )
        all_candidate_masks.append(cand)

print(f"Raw candidates across all tiles (pre-dedup): {len(scene_detections)}")

## Step 5 — Merge duplicates across tile overlaps

In [ ]:
final_detections = merge_duplicate_detections(scene_detections, iou_threshold=0.35)
print(f"Final deduplicated detections: {len(final_detections)} (ground truth: {len(scene.crowns)})")

overlay_crown_detections(
    scene.image,
    [
        {"pixel_x": d.scene_x, "pixel_y": d.scene_y, "radius_px": d.radius_px}
        for d in final_detections
    ],
    title="Final detections after duplicate merging",
)

## Step 6 — Visual embeddings + clustering

Crop each detected crown and embed it, then cluster visually similar
crowns together. This is exploratory — a precursor to supervised species
classification once labeled data exists, not a species identification
result itself.

In [ ]:
embedder = get_default_embedder()
crop_half = 20

embeddings = []
kept_detections = []

for d in final_detections:
    x, y = int(d.scene_x), int(d.scene_y)
    y0, y1 = max(0, y - crop_half), min(scene.image.shape[0], y + crop_half)
    x0, x1 = max(0, x - crop_half), min(scene.image.shape[1], x + crop_half)
    crop = scene.image[y0:y1, x0:x1]
    if crop.shape[0] < 4 or crop.shape[1] < 4:
        continue
    embeddings.append(embedder.embed(crop))
    kept_detections.append(d)

embeddings = np.stack(embeddings)
labels = cluster_embeddings(embeddings, algorithm="kmeans", n_clusters=4)
coords_2d = project_to_2d(embeddings)

plot_embedding_clusters(coords_2d, labels, title="Crown embedding clusters (exploratory)")

## Step 7 — Export to GeoJSON

Anchored at a synthetic, non-real coordinate origin — for structure
demonstration only.

In [ ]:
det_dicts = [
    {
        "pixel_x": d.scene_x,
        "pixel_y": d.scene_y,
        "species": "unknown",
        "confidence": None,
        "cluster_id": int(labels[i]),
    }
    for i, d in enumerate(kept_detections)
]

gdf = detections_to_geodataframe(det_dicts)
gdf.head()

## Summary

This notebook demonstrated the full detection-and-mapping pipeline on
synthetic data: preprocessing → tiling → segmentation → candidate
extraction → crown separation → duplicate merging → embeddings →
clustering → GIS export. No accuracy claims are made here since no
ground-truth-labeled real data is used — see `../README.md` for current
project status and roadmap.